# Practical Machine Learning — Day 02 Lab
## Data Preprocessing, Cleaning, Transformation and Pipelines
**CDAC PGCP-AI · Practical Machine Learning · Saturday 24 October 2026**

> 🔒 **TRAINER SOLUTION NOTEBOOK — do not distribute to students.**
> Same exercise IDs, same cell order and same dataset as the student notebook. Every blank is filled; every cell was executed from a fresh kernel; every number printed is computed from the data. After each exercise you will find *Syntax in simple words*, *Purpose* and *How to interpret the result*.

## Section 0 — Welcome and Instructions

### 🎯 Learning objectives
1. Diagnose data-quality problems and justify corrective actions.
2. Explain missingness, outliers and inconsistent records.
3. Choose suitable imputation and encoding strategies.
4. Understand how fitted preprocessing can cause data leakage.
5. Distinguish `fit`, `transform` and `fit_transform`.
6. Build numerical and categorical preprocessing **Pipelines**.
7. Combine branches with **ColumnTransformer**.
8. Apply preprocessing correctly to unseen observations.
9. Validate the transformed dataset and explain the results.

### ⏱️ Schedule (this notebook covers hours 4 to 7)
| Hour | Type | Content | Exercises |
|---|---|---|---|
| 4 | Guided self-learning | Data dictionary, investigation, decision log, concepts | SL1 – SL3 |
| 5 | **Lab** | Load, inspect, document, deterministic cleaning, features, split | A1 – A9, B1 – B2 |
| 6 | **Lab** | Pipelines, ColumnTransformer, validation, unseen data | B3 – B5, C1 – C5 |
| 7 | Guided self-learning | Compare strategies, debug, mini-challenges, conclusions | C6, D1 – D3, MC1 – MC3, C7 |

### 📁 Files (keep them in the same folder as this notebook)
- `06_titanic_data_dirty.csv` — the classroom dataset (**never edit or overwrite it**)
- `07_Data_Dictionary.xlsx` — column meanings, valid ranges and business rules

### ✍️ How the blanks work
- `____` marks a blank. Replace it with the missing method, value or expression. Each blank has a label such as `# TODO (A1-1)`.
- A cell with an unfinished blank **fails on purpose**. All other cells run as they are.
- 🔮 **Predict** boxes: write what you expect *before* running the cell. Comparing prediction and result is how you learn.
- Levels: **1** complete a method · **2** complete an argument or expression · **3** choose the right technique · **4** debug · **5** write it yourself.

### ▶️ Running cells
Shift + Enter. Run cells **in order**. If something looks strange: *Kernel → Restart & Run All* (Colab: *Runtime → Restart and run all*).

### 📤 Submit
1. This completed notebook 2. A cleaned, transformed dataset file is **not** required today; submit your notebook with **all outputs visible**, your Cleaning Decision Log and your final reflection (Section 12).

### ⚠️ Golden rules for today
- **Fix only what does not learn from data before splitting** (spaces, spelling, text-to-number, impossible values → missing). Anything that *learns* from the data (medians, means, categories, scaling) happens **after the split, inside a Pipeline, fitted on training data only**.
- Never edit the raw CSV. Flag before you delete. Explain every decision.
- Detailed feature engineering, feature selection, PCA and t-SNE are **Day 3**. No model is trained today.

---
# GUIDED SELF-LEARNING 1 (Hour 4, 60 min)
## Section 1 — Investigate the dataset on paper before you code
Open `07_Data_Dictionary.xlsx`. Use the sheets *Data Dictionary*, *Business Rules*, *Category Aliases* and *Dataset Notes*.

You may open the CSV in a text editor or Excel to look (read-only!), but do **not** change it.

### Exercise SL1 — Read the data dictionary
**Task:** Answer from the dictionary (no code):
1. Which column is the **target**? Which columns look like **identifiers**?
2. Which valid range is given for `Age`? Is an age of 0.42 valid? Why might it exist?
3. Which columns are **not** automatically good features, and why?
4. Which values are *invalid* and which are only *suspicious* for `Fare`?

<details><summary>💡 Hint (try first without it)</summary>

Read the sheet *Business Rules* slowly. The notes column explains why some unusual values are genuine.

</details>

**Model answers (SL1):** (1) Target: `Survived`. Identifiers: `PassengerId`, `Name`, `Ticket`. (2) Valid range 0 to 100; 0.42 is valid: infants travelled on the Titanic, so a fractional age below 1 is a genuine value, not an error. (3) `PassengerId` (row number), `Name` and `Ticket` (identifiers), `Cabin` (mostly missing). (4) Invalid: below 0 or above 1000 (the highest real ticket price was about 512). A zero fare is unusual but genuine (some crew/guarantee passengers); high first-class fares are statistical outliers but genuine.

### Exercise SL2 — Draft your cleaning decision log (plan only)
Before you see the numbers, predict what you will do. Fill the table; you will revise it in Exercise A7 and C7.

| Problem class | What I expect to find | My planned decision | Learns from data? (Yes/No) |
|---|---|---|---|
| Spaces, capitalisation, abbreviations | | | |
| Numbers stored as text | | | |
| Impossible values (age < 0 …) | | | |
| Missing numbers | | | |
| Missing categories | | | |
| Exact duplicates | | | |
| Repeated ID, different data | | | |
| Identifier columns | | | |

**Model answers (SL2):** spaces/case/abbreviations: strip, lower, dictionary map (No: deterministic). Text numbers: regex + `to_numeric` (No). Impossible values: set to NaN with a count (No: rule-based). Missing numbers: **median imputation inside a Pipeline** (**Yes**: the median is learned from training data). Missing categories: most-frequent imputation inside a Pipeline (**Yes**). Exact duplicates: drop (No). Repeated ID with conflicting data: quarantine all rows of that ID (No) so one passenger cannot be in both train and test. Identifier columns: exclude from X (No).

**Key idea to stress:** the last column is the leakage test. *If a step learns a number from the data, it must be fitted on training data only.*

### Exercise SL3 — Conceptual questions (write 1–3 sentences each)
1. Why can imputing missing ages with the median of the **whole** dataset before splitting cause leakage?
2. What is the difference between `fit`, `transform` and `fit_transform`?
3. Why do we use `handle_unknown="ignore"` in a one-hot encoder?
4. What does a `Pipeline` guarantee that separate manual steps do not?
5. What does `ColumnTransformer` do and why do we need it?
6. Why is a Fare of 512 not automatically an error, but a Fare of 5123 is?

**Model answers (SL3):** (1) The median would use information from the test rows, so the test set is no longer fully unseen; test performance would look slightly better than reality. (2) `fit` learns parameters (e.g. mean, median, categories) from data; `transform` applies already-learned parameters; `fit_transform` does both in one call and must only be used on training data. (3) A category may appear in new data that was not in the training data (e.g. a new port); without `ignore` the encoder raises an error; with `ignore` it outputs all zeros for that feature. (4) That the same ordered steps are fitted together on training data and applied identically to any later data. (5) It applies different Pipelines to different column groups (numeric vs categorical) and joins the outputs. (6) 512 matches the highest real ticket price (first-class suite) so it is a genuine outlier; 5123 exceeds anything possible (decimal slip), so it breaks a business rule.

---
# STAGE A — Investigate and Prepare Data (Hour 5)
## Section 2 — Load and inspect

### Exercise A1: Load the CSV

**Level 1** &nbsp;|&nbsp; **Task:** Read `06_titanic_data_dirty.csv` into a DataFrame called `df`.

**You should see:** `Loaded: 06_titanic_data_dirty.csv`.

<details><summary>💡 Hint (try first without it)</summary>

Pandas has one `read_` function per file type.

</details>

In [1]:
# ===== Exercise A1 =====
import os, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)

# Optional (Google Colab only): upload the CSV. Outside Colab this does nothing.
try:
    from google.colab import files
    if not os.path.exists("06_titanic_data_dirty.csv"):
        files.upload()
except ImportError:
    pass

DATA_FILE = "06_titanic_data_dirty.csv"
assert os.path.exists(DATA_FILE), f"{DATA_FILE} not found in {os.getcwd()}"

# Fingerprint of the raw file: at the end we prove it was never modified
RAW_FINGERPRINT = hashlib.md5(open(DATA_FILE, "rb").read()).hexdigest()

# TODO (A1-1): complete the pandas function that reads a CSV file
df = pd.read_csv(DATA_FILE)
print("Loaded:", DATA_FILE)

Loaded: 06_titanic_data_dirty.csv


**Syntax in simple words:** `pd.read_csv(path)` reads a comma-separated file into a table (DataFrame). The MD5 line makes a short fingerprint of the file's bytes.

**Purpose:** Load the raw data and record a fingerprint so we can prove at the end that the raw file was never changed.

**How to interpret the result:** Printing 'Loaded' means the file was found and parsed; the fingerprint is checked again in the last section.

### Exercise A2: Shape, types and first rows

**Level 1** &nbsp;|&nbsp; **Task:** Display the data types and non-null counts, then look at the first rows.

**You should see:** A summary of 12 columns. **Look at the Dtype of `Age` and `Fare`.**

<details><summary>💡 Hint (try first without it)</summary>

The method name is a synonym for 'information'.

</details>

**🤔 Reflect (A2):** Which columns that should be numeric are stored as text, and what might cause this?

In [2]:
# ===== Exercise A2 =====
print("Rows, columns:", df.shape)
# TODO (A2-1): display column dtypes and non-null counts
df.info()
df.head(8)

Rows, columns: (616, 12)
<class 'pandas.DataFrame'>
RangeIndex: 616 entries, 0 to 615
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   PassengerId  616 non-null    int64
 1   Survived     613 non-null    str  
 2   Pclass       610 non-null    str  
 3   Name         616 non-null    str  
 4   Sex          611 non-null    str  
 5   Age          499 non-null    str  
 6   SibSp        616 non-null    int64
 7   Parch        616 non-null    int64
 8   Ticket       616 non-null    str  
 9   Fare         604 non-null    str  
 10  Cabin        142 non-null    str  
 11  Embarked     607 non-null    str  
dtypes: int64(3), str(9)
memory usage: 57.9 KB


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.25,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38,1,0,PC 17599,71.2833,C85,C
2,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1,C123,S
3,5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.05,NaN,S
4,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
5,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,-4,1,0,237736,30.0708,NaN,C
6,11,1,3,"Sandstrom, Miss Marguerite Rut",female,4,1,1,PP 9549,16.7,G6,s
7,13,0,3,"Saundercock, Mr. William Henry",male,20,0,0,A/5. 2151,8.05,NaN,S


**Syntax in simple words:** `df.shape` is a pair (rows, columns). `df.info()` lists each column's type and how many cells are filled.

**Purpose:** Understand the structure before changing anything.

**How to interpret the result:** `Age` and `Fare` are text because a few cells contain symbols or words (such as `22 yrs` or `£7.25`); one such cell turns the whole column into text.

## Section 3 — Missing values (A3)

### Exercise A3: Count missing values

**Level 1** &nbsp;|&nbsp; **Task:** Count missing cells per column and as a percentage.

**You should see:** A table sorted from most to least missing. Which column is almost empty, and what does that suggest about using it as a feature?

<details><summary>💡 Hint (try first without it)</summary>

A method that returns True for missing cells (two names exist). `True` counts as 1 in `.sum()`.

</details>

In [3]:
# ===== Exercise A3 =====
# TODO (A3-1): True where a value is missing, then add up per column
missing = df.isnull().sum()
missing_report = pd.DataFrame({"missing_count": missing, "missing_percent": (missing / len(df) * 100).round(1)})
missing_report.sort_values("missing_count", ascending=False)

,missing_count,missing_percent
Cabin,474,76.9
Age,117,19.0
Fare,12,1.9
Embarked,9,1.5
Pclass,6,1.0
Sex,5,0.8
Survived,3,0.5
PassengerId,0,0.0
Name,0,0.0
SibSp,0,0.0


**Syntax in simple words:** `df.isnull()` gives a True/False table; `.sum()` adds the Trues per column.

**Purpose:** Quantify missing data. Decide later whether a column is worth keeping and how to fill gaps.

**How to interpret the result:** A column that is mostly empty carries little usable information; a column with a few gaps can be imputed.

In [4]:
# Model answer (computed)
worst = missing_report["missing_count"].idxmax()
print(f"Most missing: {worst} ({missing_report.loc[worst,'missing_count']} rows, {missing_report.loc[worst,'missing_percent']}%)")
print("Columns with missing values:", list(missing_report[missing_report.missing_count > 0].index))

Most missing: Cabin (474 rows, 76.9%)
Columns with missing values: ['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Cabin', 'Embarked']


## Section 4 — Duplicates and repeated IDs (A4)

### Exercise A4: Exact duplicates versus repeated PassengerId

**Level 2** &nbsp;|&nbsp; **Task:** (a) Count exact duplicate rows. (b) Mark every row whose `PassengerId` appears more than once. (c) Find which repeated IDs have **conflicting** values.

**You should see:** A count, a table of repeated-ID rows and a list of conflicting IDs.

<details><summary>💡 Hint (try first without it)</summary>

(a) The method name is a synonym for 'repeated'. (b) `value_counts()` says how often each ID appears: which count means 'repeated'?

</details>

**🤔 Reflect (A4):** If the same passenger appeared in both the training set and the test set, what would happen to your test score?

In [5]:
# ===== Exercise A4 =====
# (a) TODO (A4-1): count rows that are exact copies of an earlier row
exact_dups = df.duplicated().sum()
print("Exact duplicate rows:", exact_dups)

# (b) how many times does each PassengerId appear?
id_counts = df["PassengerId"].value_counts()
# TODO (A4-2): keep the IDs that appear MORE than this many times (what count means "repeated"?)
repeated_ids = id_counts[id_counts > 1].index
repeated_rows = df[df["PassengerId"].isin(repeated_ids)].sort_values("PassengerId")
print("Rows whose PassengerId appears more than once:", len(repeated_rows))

# (c) provided: count distinct values per column inside each ID. 2+ distinct values = the rows disagree
distinct_per_id = df.groupby("PassengerId").nunique(dropna=False)
conflicting_ids = sorted(distinct_per_id[(distinct_per_id > 1).any(axis=1)].index)
print("IDs with CONFLICTING data:", conflicting_ids)
repeated_rows[repeated_rows["PassengerId"].isin(conflicting_ids)][["PassengerId", "Survived", "Pclass", "Sex", "Age", "Fare"]]

Exact duplicate rows: 10
Rows whose PassengerId appears more than once: 32
IDs with CONFLICTING data: [21, 300, 435, 489, 670, 716]


,PassengerId,Survived,Pclass,Sex,Age,Fare
12,21,0,2,male,35,26
265,21,0,3,male,29,57.2
162,300,1,2,female,44,123.76
208,300,1,1,female,50,247.521
531,435,0,1,male,54,122.98
300,435,0,1,male,50,55.9
336,489,0,3,male,30,8.05
33,489,0,3,male,24,17.71
495,670,1,1,female,NaN,83.2
463,670,1,1,female,NaN,52


**Syntax in simple words:** `df.duplicated()` is True for a row identical to an earlier row. `value_counts()` counts how often each ID appears; keeping counts above 1 gives the repeated IDs. `groupby(...).nunique()` counts distinct values per column inside each ID.

**Purpose:** Separate harmless exact copies (safe to drop) from conflicting records (we cannot know which is right).

**How to interpret the result:** Conflicting IDs are the dangerous ones: the same passenger could land in both train and test with different values. Decision on A8: quarantine them.

## Section 5 — Inconsistent categories (A5)

### Exercise A5: Inspect the categorical columns

**Level 1** &nbsp;|&nbsp; **Task:** Print the value counts (including missing) for `Sex`, `Embarked`, `Pclass` and `Survived`.

**You should see:** Several spellings of the same value, abbreviations and some unexpected values.

<details><summary>💡 Hint (try first without it)</summary>

Missing values are hidden by default (`dropna=True`). Which boolean shows them?

</details>

**🤔 Reflect (A5):** Which values need a **dictionary map**, and which should simply become missing?

In [6]:
# ===== Exercise A5 =====
for col in ["Sex", "Embarked", "Pclass", "Survived"]:
    print(f"--- {col} ---")
    # TODO (A5-1): include the missing values in the count (dropna=True would hide them)
    print(df[col].value_counts(dropna=False))
    print()

--- Sex ---
Sex
male       374
female     201
M            6
FEMALE       6
NaN          5
male         4
MALE         4
Unknown      3
 male        3
Male         3
F            2
Female       2
 female      2
female       1
Name: count, dtype: int64

--- Embarked ---
Embarked
S              428
C              103
Q               44
Southampton     10
s                9
NaN              9
S                7
X                3
c                1
Queenstown       1
Cherbourg        1
Name: count, dtype: int64

--- Pclass ---
Pclass
3       323
1       141
2       132
NaN       6
3rd       5
1st       5
2nd       2
Crew      2
Name: count, dtype: int64

--- Survived ---
Survived
0      384
1      223
No       4
NaN      3
Yes      2
Name: count, dtype: int64



**Syntax in simple words:** `value_counts()` lists each distinct value and how many rows have it; `dropna=False` also counts missing cells (you met `value_counts` in A4).

**Purpose:** Find spelling variants, abbreviations, full names and unexpected categories.

**How to interpret the result:** Group variants that mean the same thing (male / Male / M); values that mean nothing (Unknown, X, Crew) become missing.

## Section 6 — Numbers stored as text, invalid values and outliers (A6)

### Exercise A6: Convert, apply business rules and compare with the IQR rule

**Level 2** &nbsp;|&nbsp; **Task:** Convert `Age` and `Fare` to numbers; flag values that break the **business rules**; then compare with the statistical **IQR outlier rule** for `Fare`.

**You should see:** Counts of invalid ages and fares, and a much larger count of IQR outliers.

<details><summary>💡 Hint (try first without it)</summary>

`to_numeric` with `errors='coerce'`. The age limit is on the Business Rules sheet. The classic IQR fence multiplier is a number between 1 and 2.

</details>

**🤔 Reflect (A6):** Why is a statistical outlier not always an error? Which fares are genuine?

In [7]:
# ===== Exercise A6 =====
def to_number(series):
    """Keep digits, point and minus sign; convert; unreadable text becomes NaN."""
    only_number = series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True)
    # TODO (A6-1): convert the text to numbers; unreadable values must become NaN
    return pd.to_numeric(only_number, errors="coerce")

age_num, fare_num = to_number(df["Age"]), to_number(df["Fare"])

# Business rules (07_Data_Dictionary.xlsx -> Business Rules)
# TODO (A6-2): the largest valid age
age_invalid = (age_num < 0) | (age_num > 100)
fare_invalid = (fare_num < 0) | (fare_num > 1000)
print("Invalid ages :", int(age_invalid.sum()))
print("Invalid fares:", int(fare_invalid.sum()))
print("Ages below 1 (infants):", int(((age_num > 0) & (age_num < 1)).sum()), "| zero fares:", int((fare_num == 0).sum()))

# Statistical outliers on the VALID fares (IQR rule)
fare_valid = fare_num[(fare_num >= 0) & (fare_num <= 1000)]
q1, q3 = fare_valid.quantile(0.25), fare_valid.quantile(0.75)
iqr = q3 - q1
# TODO (A6-3): the usual fence multiplier of the IQR rule
upper_fence = q3 + 1.5 * iqr
iqr_outliers = fare_valid[fare_valid > upper_fence]
print(f"IQR upper fence: {upper_fence:.1f} | valid fares above it: {len(iqr_outliers)} | highest valid fare: {fare_valid.max():.1f}")

Invalid ages : 4
Invalid fares: 6
Ages below 1 (infants): 4 | zero fares: 9
IQR upper fence: 65.6 | valid fares above it: 79 | highest valid fare: 512.3


**Syntax in simple words:** The regex keeps only digits, `.` and `-`; `to_numeric(errors='coerce')` converts and turns anything unreadable into NaN. A rule such as `(x < 0) | (x > 1000)` is True for impossible values. The IQR rule marks values above `Q3 + 1.5 x IQR`.

**Purpose:** Separate three things that look alike: **invalid** (breaks a rule), **suspicious** (possible but unusual), **statistical outlier** (far from the rest but possibly genuine).

**How to interpret the result:** Business rules flag only a handful of impossible fares, while the IQR rule flags many more, mostly genuine first-class tickets. Rules decide *errors*; statistics only suggest *where to look*.

## Section 7 — Cleaning decision log (A7)

### Exercise A7 — Cleaning plan
**Task:** Complete the table. For each problem you found in A3 – A6 write the **evidence** (a number from your outputs), the **decision**, and whether it must be done *before* or *inside the pipeline after* the split.

| Problem | Evidence (number from your output) | Decision | Before split (deterministic) or inside pipeline (learns)? | Risk |
|---|---|---|---|---|
| Mostly empty column | | | | |
| Missing Age / Fare | | | | |
| Missing Sex / Embarked / Pclass | | | | |
| Spelling variants and aliases | | | | |
| Text numbers (`22 yrs`, `£7.25`) | | | | |
| Impossible values | | | | |
| Suspicious ages 90-100 | | | | |
| Exact duplicates | | | | |
| Conflicting repeated IDs | | | | |
| Rows with no `Survived` | | | | |

<details><summary>💡 Hint (try first without it)</summary>

Use the rule of thumb: *if the decision needs a number computed from the data (median, most frequent, mean), it goes inside the pipeline.*

</details>

**Model decision log (A7)** — evidence numbers are printed by the cells above and listed in the Trainer Key (Section D).

| Problem | Decision | Before split or inside pipeline? | Risk |
|---|---|---|---|
| Cabin mostly empty | exclude from X | before | loses a possible signal (Day 3 may extract a feature) |
| Missing Age / Fare | median imputation | **inside pipeline** | assumes values missing at random; reduces variance |
| Missing Sex / Embarked / Pclass | most-frequent imputation | **inside pipeline** | may bias towards the majority class |
| Spelling / aliases | strip, lower, dictionary map | before | unmapped variants become missing |
| Text numbers | regex + `to_numeric` | before | coercion hides unreadable text as NaN: count it |
| Impossible values | set to NaN (counted) | before | fills later with estimates |
| Suspicious ages 90-100 | flag, keep | before | may be errors; effect is small |
| Exact duplicates | drop | before | none |
| Conflicting IDs | quarantine all rows of those IDs | before | loses 12 genuine rows; avoids train/test leakage |
| No `Survived` | remove from modelling table | before | none: no label, no supervised learning |

## Section 8 — Safe, deterministic cleaning (A8)

> **Safe** means: the step does **not** learn anything from the data. Replacing `MALE` by `male` or `-4` by missing is the same no matter which rows you look at. Computing a median **is not safe** before the split, so we leave it for the pipeline.

### Exercise A8: Clean text, numbers, duplicates, conflicts and unlabelled rows

**Level 2** &nbsp;|&nbsp; **Task:** Standardise the text columns, convert the numbers, turn impossible values into missing, remove exact duplicates and quarantine conflicting IDs and rows without a label.

**You should see:** A row-accounting table that adds up, and cleaned categories: `Sex` has male/female (+NaN), `Embarked` has S/C/Q (+NaN), `Pclass` has First/Second/Third (+NaN).

<details><summary>💡 Hint (try first without it)</summary>

strip → lower → dictionary `map` (unmapped values become NaN, provided); `drop_duplicates()` for exact copies; `np.nan` for impossible values.

</details>

**🤔 Reflect (A8):** Why do we quarantine **all** rows of a conflicting ID instead of keeping the first one?

In [8]:
# ===== Exercise A8 =====
clean_df = df.copy()

# (1) remove the spaces around the text values
for col in ["Sex", "Embarked", "Pclass", "Survived"]:
    # TODO (A8-1): remove spaces at the start and end of every value
    clean_df[col] = clean_df[col].str.strip()

# (2) make the values lower-case, then replace each by its standard label. Values not in a dictionary become NaN.
sex_map      = {"male": "male", "m": "male", "female": "female", "f": "female"}
embarked_map = {"s": "S", "southampton": "S", "c": "C", "cherbourg": "C", "q": "Q", "queenstown": "Q"}
pclass_map   = {"1": "First", "1st": "First", "2": "Second", "2nd": "Second", "3": "Third", "3rd": "Third"}
survived_map = {"0": 0, "1": 1, "no": 0, "yes": 1}
for col, mapping in {"Sex": sex_map, "Embarked": embarked_map, "Pclass": pclass_map, "Survived": survived_map}.items():
    # TODO (A8-2): write every value in lower-case (the dictionary keys are lower-case)
    clean_df[col] = clean_df[col].str.lower().map(mapping)

# (3) numbers stored as text -> numbers (to_number comes from A6)
for col in ["Age", "Fare"]:
    clean_df[col] = to_number(clean_df[col])

# (4) impossible values -> missing. Rules come from the Business Rules sheet. We count what we change.
invalid_rules = {"Age": lambda s: (s < 0) | (s > 100), "Fare": lambda s: (s < 0) | (s > 1000),
                 "SibSp": lambda s: (s < 0) | (s > 10), "Parch": lambda s: (s < 0) | (s > 10)}
invalid_counts = {}
for col, rule in invalid_rules.items():
    bad = rule(clean_df[col])
    invalid_counts[col] = int(bad.sum())
    # TODO (A8-4): set the impossible values to missing (use the NumPy missing-value constant)
    clean_df.loc[bad, col] = np.nan
print("Impossible values set to missing:", invalid_counts)

# (5) exact duplicates
rows_start = len(clean_df)
# TODO (A8-3): drop rows that are exact copies of an earlier row (keep the first)
clean_df = clean_df.drop_duplicates()
n_exact = rows_start - len(clean_df)

# (6) conflicting IDs: quarantine ALL rows of those IDs
conflict_mask = clean_df["PassengerId"].duplicated(keep=False)
quarantine_conflicts = clean_df[conflict_mask]
clean_df = clean_df[~conflict_mask]

# (7) rows without a label cannot be used for supervised learning
no_label = clean_df["Survived"].isna()
quarantine_no_label = clean_df[no_label]
clean_df = clean_df[~no_label].reset_index(drop=True)
clean_df["Survived"] = clean_df["Survived"].astype(int)

accounting = pd.DataFrame({"step": ["Raw rows", "Exact duplicates removed", "Conflicting-ID rows quarantined", "Rows without Survived quarantined", "Rows in modelling table"],
                           "rows": [len(df), n_exact, len(quarantine_conflicts), len(quarantine_no_label), len(clean_df)]})
assert len(df) == n_exact + len(quarantine_conflicts) + len(quarantine_no_label) + len(clean_df), "Row accounting does not add up"
print(accounting.to_string(index=False))
for col in ["Sex", "Embarked", "Pclass"]:
    print(clean_df[col].value_counts(dropna=False).to_dict())

Impossible values set to missing: {'Age': 4, 'Fare': 6, 'SibSp': 4, 'Parch': 2}
                             step  rows
                         Raw rows   616
         Exact duplicates removed    10
  Conflicting-ID rows quarantined    12
Rows without Survived quarantined     3
          Rows in modelling table   591
{'male': 381, 'female': 202, nan: 8}
{'S': 434, 'C': 101, 'Q': 45, nan: 11}
{'Third': 320, 'First': 134, 'Second': 129, nan: 8}


**Syntax in simple words:** `str.strip()` trims spaces, `str.lower()` lower-cases, `map(dict)` swaps each value for its dictionary value (NaN if absent). `df.loc[mask, col] = np.nan` writes a missing value where the mask is True. `duplicated(keep=False)` selects every row of a repeated ID, `~mask` means NOT.

**Purpose:** Apply only **rule-based** fixes before the split, and keep a row-accounting table so every removed row is explained.

**How to interpret the result:** The assertion proves no row vanished without a reason. Note that `Age` and `Fare` still contain NaN: they are filled later, inside the Pipeline.

## Section 9 — Choose the features (A9)

### Exercise A9: Select the candidate features

**Level 2** &nbsp;|&nbsp; **Task:** Remove the columns that should **not** be automatically used as features.

**You should see:** A table `model_df` with `Survived` plus 7 candidate features.

<details><summary>💡 Hint (try first without it)</summary>

Name the identifiers and the mostly empty column in a list and use the DataFrame method that removes columns.

</details>

**🤔 Reflect (A9):** Why would a model that uses `PassengerId` or `Ticket` be risky?

In [9]:
# ===== Exercise A9 =====
# TODO (A9-1): list the columns to exclude: row id, free-text name, ticket text, mostly-empty cabin
exclude = ["PassengerId", "Name", "Ticket", "Cabin"]
model_df = clean_df.drop(columns=exclude)
print("Columns kept:", list(model_df.columns))
model_df.head()

Columns kept: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,Third,male,22.0,1.0,0.0,7.2500,S
1,1,First,female,38.0,1.0,0.0,71.2833,C
2,1,First,female,35.0,1.0,0.0,53.1000,S
3,0,Third,male,35.0,0.0,0.0,8.0500,S
4,0,Third,male,NaN,0.0,0.0,8.4583,Q


**Syntax in simple words:** A Python list in square brackets names the columns; `drop(columns=list)` returns a table without them.

**Purpose:** Keep only columns that carry real information. Identifiers can let a model memorise passengers instead of learning patterns; Cabin is mostly empty.

**How to interpret the result:** Seven candidate features remain: Pclass, Sex, Age, SibSp, Parch, Fare, Embarked. Turning Cabin or Name into smarter features is Day 3.

---
# STAGE B — Split Data and Build Transformations (Hours 5 and 6)
## Section 10 — Features, target and the split

### Exercise B1: Separate features and target

**Level 1** &nbsp;|&nbsp; **Task:** Create the feature table `X` and the target `y`.

**You should see:** `X` with 7 columns and `y` with only 0 and 1.

<details><summary>💡 Hint (try first without it)</summary>

`X` is everything except the target; `y` is the one column you want to predict (use its name in quotes).

</details>

In [10]:
# ===== Exercise B1 =====
target = "Survived"
X = model_df.drop(columns=[target])
# TODO (B1-1): the target column as a Series (name of the column in quotes)
y = model_df["Survived"]
print("X shape:", X.shape, "| y shape:", y.shape)
print("Features:", list(X.columns))

X shape: (591, 7) | y shape: (591,)
Features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


**Syntax in simple words:** `model_df[name]` selects one column (a Series). `drop(columns=[...])` removes columns.

**Purpose:** Features (inputs) and target (answer) must be separate before splitting.

**How to interpret the result:** `X` and `y` must have the same number of rows.

### Exercise B2: Split into training and test sets

**Level 2** &nbsp;|&nbsp; **Task:** Split with 20% for testing, keeping the survivor share equal in both parts, with a fixed random seed.

**You should see:** Four shapes and a table showing nearly equal survival rates in train and test.

<details><summary>💡 Hint (try first without it)</summary>

`test_size` is a fraction; `stratify` needs the target to keep class proportions.

</details>

**🤔 Reflect (B2):** Why must the split happen *before* any imputation or scaling?

In [11]:
# ===== Exercise B2 =====
from sklearn.model_selection import train_test_split

# TODO (B2-1): hold out 20% of the rows for testing (give a fraction)
# TODO (B2-2): keep the survival proportion equal in both parts (stratify by the target)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Survival rate - train: %.3f | test: %.3f" % (y_train.mean(), y_test.mean()))

Train: (472, 7) | Test: (119, 7)
Survival rate - train: 0.367 | test: 0.370


**Syntax in simple words:** `train_test_split` shuffles and cuts the rows. `test_size=0.2` puts 20% in the test set; `stratify=y` keeps the share of 0s and 1s equal; `random_state=42` makes the shuffle repeatable.

**Purpose:** Create a test set that stays **unseen** until preprocessing has been fitted on training data.

**How to interpret the result:** The two survival rates are almost identical (stratification). From here on, anything that learns a statistic uses `X_train` only.

## Section 11 — `fit`, `transform` and pipelines (Hour 6)

### Exercise B3: See fit and transform with your own eyes

**Level 1** &nbsp;|&nbsp; **Task:** Fit a `StandardScaler` on the training `Fare`, read what it learned, then transform the test `Fare`.

**You should see:** The learned mean of the training Fare, and scaled test values whose mean is **not exactly** 0.

<details><summary>💡 Hint (try first without it)</summary>

`fit` learns, `transform` applies. The learned mean is stored in an attribute ending with an underscore.

</details>

**🤔 Reflect (B3):** What would change if you called `fit` on the test set instead?

In [12]:
# ===== Exercise B3 =====
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# TODO (B3-1): LEARN the mean and standard deviation from the TRAINING fares only
scaler.fit(X_train[["Fare"]])

# TODO (B3-2): the attribute holding the mean learned from the training data
print("Mean learned from training Fare:", scaler.mean_)

# TODO (B3-3): APPLY the already-learned rule to the TEST fares (no re-learning)
test_scaled = scaler.transform(X_test[["Fare"]])
print("Mean of scaled TEST Fare (missing values ignored):", float(np.nanmean(test_scaled)))
print("(By construction the mean of the scaled TRAIN Fare would be 0.)")

Mean learned from training Fare: [33.06821007]
Mean of scaled TEST Fare (missing values ignored): -0.030165125044562542
(By construction the mean of the scaled TRAIN Fare would be 0.)


**Syntax in simple words:** `scaler.fit(data)` stores the mean and standard deviation of `data` inside the scaler (attributes ending in `_`). `scaler.transform(data)` computes `(x - mean) / std` using the *stored* numbers. Missing values stay missing.

**Purpose:** Show that scaling is a learned rule: the rule comes from the training data and is then applied unchanged to the test data.

**How to interpret the result:** Train mean after scaling is 0 by construction; the test mean is close to, but not exactly, 0 — that small difference is the honest sign that the test set was not used for learning.

### Exercise B4: Numeric pipeline

**Level 3** &nbsp;|&nbsp; **Task:** Build a Pipeline for the numeric columns: fill missing values with a robust statistic, then standardise.

**You should see:** A pipeline object listing two named steps.

<details><summary>💡 Hint (try first without it)</summary>

Which central statistic is robust to extreme fares? And which class standardises?

</details>

**🤔 Reflect (B4):** Why does the order impute-then-scale matter?

In [13]:
# ===== Exercise B4 =====
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked"]

numeric_pipeline = Pipeline(steps=[
    # TODO (B4-1): fill missing numbers with the median (choose the strategy name)
    ("impute", SimpleImputer(strategy="median")),
    # TODO (B4-2): choose the scaler that gives mean 0 and standard deviation 1
    ("scale", StandardScaler()),
])
numeric_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('impute', ...), ('scale', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a fea

**Syntax in simple words:** `Pipeline(steps=[(name, step), ...])` chains steps: the output of one is the input of the next. `SimpleImputer(strategy='median')` replaces NaN by the column median learned during `fit`.

**Purpose:** One object that learns and applies a whole sequence consistently. Fitting the pipeline learns the median **and then** the mean/std of the *imputed* column.

**How to interpret the result:** Impute first (scalers cannot learn mean/std properly from gaps), then scale.

### Exercise B5: Categorical pipeline

**Level 3** &nbsp;|&nbsp; **Task:** Build a Pipeline for the categorical columns: fill missing categories with the most frequent value, then one-hot encode — without crashing on unseen categories.

**You should see:** A pipeline object listing two named steps.

<details><summary>💡 Hint (try first without it)</summary>

Strategy name: the *most frequent* category. For unseen categories the encoder needs `handle_unknown` set to the word that means 'do not raise an error'.

</details>

**🤔 Reflect (B5):** What will the encoder output for a category it has never seen?

In [14]:
# ===== Exercise B5 =====
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline(steps=[
    # TODO (B5-1): fill missing categories with the most frequent value (strategy name)
    ("impute", SimpleImputer(strategy="most_frequent")),
    # TODO (B5-2): do NOT crash when a category appears that was not seen in training
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
categorical_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('impute', ...), ('onehot', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. 

**Syntax in simple words:** `most_frequent` fills with the commonest category in the training data. `handle_unknown='ignore'` makes unknown categories encode as all zeros instead of raising an error. `sparse_output=False` returns an ordinary table.

**Purpose:** Prepare categories so that missing values and unseen values cannot break the pipeline at prediction time.

**How to interpret the result:** The one-hot columns are created from the categories seen in training; a new category simply has all zeros in those columns.

---
# STAGE C — Combine, Validate and Interpret (Hour 6)
## Section 12 — ColumnTransformer

### Exercise C1: Combine the branches

**Level 3** &nbsp;|&nbsp; **Task:** Combine the numeric and categorical pipelines in one `ColumnTransformer`.

**You should see:** A diagram-like object with two branches: `num` and `cat`.

<details><summary>💡 Hint (try first without it)</summary>

The class name is exactly what it does: it *transforms columns*, a different transformer for each group.

</details>

**🤔 Reflect (C1):** What would happen to a column you do not list? (check the `remainder` argument)

In [15]:
# ===== Exercise C1 =====
import sklearn.compose as skc

# TODO (C1-1): the class of this module that applies different pipelines to different groups of columns
preprocess = skc.ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
], remainder="drop")
preprocess

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

**Syntax in simple words:** Each tuple is (name, transformer, columns). Columns not listed are dropped because `remainder='drop'`.

**Purpose:** One preprocessing object for the whole table: the only thing we need to fit and reuse later.

**How to interpret the result:** If a feature name is misspelt, sklearn raises a KeyError telling you which column is missing.

### Exercise C2: Fit on training data, transform both

**Level 1** &nbsp;|&nbsp; **Task:** Fit the preprocessing on `X_train` and transform `X_train` and `X_test`.

**You should see:** Two matrices with the **same number of columns** and different numbers of rows.

<details><summary>💡 Hint (try first without it)</summary>

Training: `fit` and `transform` in one call. Test: only `transform`.

</details>

**🤔 Reflect (C2):** Why is calling `fit_transform` on the test set a mistake?

In [16]:
# ===== Exercise C2 =====
# TODO (C2-1): LEARN from the training data and transform it in one call
X_train_prep = preprocess.fit_transform(X_train)
# TODO (C2-2): APPLY the learned rules to the test data (no learning!)
X_test_prep = preprocess.transform(X_test)

print("Train matrix:", X_train_prep.shape, "| Test matrix:", X_test_prep.shape)
imputer = preprocess.named_transformers_["num"].named_steps["impute"]
print("Medians learned from TRAIN:", dict(zip(numeric_features, imputer.statistics_.round(2).tolist())))

Train matrix: (472, 12) | Test matrix: (119, 12)
Medians learned from TRAIN: {'Age': 28.75, 'SibSp': 0.0, 'Parch': 0.0, 'Fare': 14.45}


**Syntax in simple words:** `fit_transform(X_train)` learns the medians, most-frequent categories, means, standard deviations and category lists **from the training data**, then returns the transformed training matrix. `transform(X_test)` reuses those learned numbers.

**Purpose:** Guarantee that the test set never influences preprocessing.

**How to interpret the result:** Same column count for train and test; the medians printed come from training rows only.

### Exercise C3: Feature names and dimensions

**Level 1** &nbsp;|&nbsp; **Task:** Turn the matrix back into a labelled table.

**You should see:** A table whose columns are named like `num__Age`, `cat__Sex_male`.

<details><summary>💡 Hint (try first without it)</summary>

A method of the fitted ColumnTransformer that returns the output column names; its name starts with `get_feature_names_out`.

</details>

**🤔 Reflect (C3):** Which input column produced the most output columns, and why?

In [17]:
# ===== Exercise C3 =====
# TODO (C3-1): ask the fitted preprocessor for the names of the output columns
feature_names = preprocess.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_prep, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_prep, columns=feature_names, index=X_test.index)
print("Output columns:", len(feature_names))
print(list(feature_names))
X_train_df.head()

Output columns: 12
['num__Age', 'num__SibSp', 'num__Parch', 'num__Fare', 'cat__Pclass_First', 'cat__Pclass_Second', 'cat__Pclass_Third', 'cat__Sex_female', 'cat__Sex_male', 'cat__Embarked_C', 'cat__Embarked_Q', 'cat__Embarked_S']


,num__Age,num__SibSp,num__Parch,num__Fare,cat__Pclass_First,cat__Pclass_Second,cat__Pclass_Third,cat__Sex_female,cat__Sex_male,cat__Embarked_C,cat__Embarked_Q,cat__Embarked_S
24,-1.176464,0.491891,-0.451788,-0.399047,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
46,-0.671408,-0.483624,-0.451788,0.770909,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
234,0.843760,-0.483624,-0.451788,-0.447515,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
54,-0.960011,-0.483624,-0.451788,-0.412985,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
164,-0.022050,-0.483624,-0.451788,-0.474528,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0


**Syntax in simple words:** `get_feature_names_out()` gives each output column a name (`branch__column_value`). We wrap the matrix in a DataFrame with these names.

**Purpose:** Be able to read the transformed data: which column is which.

**How to interpret the result:** 4 numeric columns + one-hot columns per category seen in training (Pclass 3, Sex 2, Embarked 3). That is why the width grows from 7 inputs.

### Exercise C4: Validate the transformed data

**Level 5** &nbsp;|&nbsp; **Task:** Run the provided checks, then **write three checks yourself**.

**You should see:** All provided checks pass, then your own pass too.

<details><summary>💡 Hint (try first without it)</summary>

Ideas: same width; no NaN anywhere; one-hot columns for one feature add up to 1 in every row (e.g. the `cat__Sex_` columns).

</details>

**🤔 Reflect (C4):** Why is it correct that test numeric means are not exactly 0?

In [18]:
# ===== Exercise C4 (provided checks) =====
assert X_train_df.shape[1] == X_test_df.shape[1], "Train and test have different columns"
assert X_train_df.index.is_unique and X_test_df.index.is_unique
assert set(X_train_df.index).isdisjoint(set(X_test_df.index)), "Same row in train and test!"
num_cols = [c for c in X_train_df.columns if c.startswith("num__")]
assert np.allclose(X_train_df[num_cols].mean(), 0, atol=1e-6), "Train numeric means should be ~0"
assert np.allclose(X_train_df[num_cols].std(ddof=0), 1, atol=1e-6), "Train numeric std should be ~1"
print("Provided checks passed")
print("Test numeric means (small but not exactly 0):")
print(X_test_df[num_cols].mean().round(3).to_dict())

Provided checks passed
Test numeric means (small but not exactly 0):
{'num__Age': 0.048, 'num__SibSp': 0.098, 'num__Parch': -0.039, 'num__Fare': -0.028}


In [19]:
# ===== Exercise C4-L5 (Level 5) - SOLUTION =====
assert not X_train_df.isnull().any().any(), "Missing values in the training matrix"
assert not X_test_df.isnull().any().any(), "Missing values in the test matrix"
sex_cols = [c for c in X_train_df.columns if c.startswith("cat__Sex_")]
assert (X_train_df[sex_cols].sum(axis=1) == 1).all(), "Sex one-hot group does not add up to 1 (train)"
assert (X_test_df[sex_cols].sum(axis=1) == 1).all(), "Sex one-hot group does not add up to 1 (test)"
assert len(X_train_df) == len(X_train) and len(X_test_df) == len(X_test), "Row counts changed"
print("Student-written checks passed")

Student-written checks passed


**Syntax in simple words:** `assert condition, message` stops with the message if the condition is False. `np.allclose(a, b, atol)` compares numbers allowing a tiny error.

**Purpose:** Prove the transformed data is sound instead of hoping it is.

**How to interpret the result:** Train numeric means are 0 and std 1 by construction. Test means are close to 0 but not exactly 0: the rule was learned on training data only.

### Exercise C5: Transform an unseen passenger

**Level 1** &nbsp;|&nbsp; **Task:** Run a brand-new passenger through the **already fitted** preprocessing.

**You should see:** One row with the median age (scaled), and zeros in all Embarked columns.

<details><summary>💡 Hint (try first without it)</summary>

The new row needs only `transform`; the fitted preprocessing remembers everything.

</details>

**🤔 Reflect (C5):** What would have happened without `handle_unknown='ignore'`?

In [20]:
# ===== Exercise C5 =====
new_passenger = pd.DataFrame([{"Pclass": "Second", "Sex": "female", "Age": np.nan, "SibSp": 1, "Parch": 0, "Fare": 26.0, "Embarked": "X"}])
# TODO (C5-1): apply the fitted preprocessing to the new passenger (no re-learning)
new_prep = preprocess.transform(new_passenger)
new_df = pd.DataFrame(new_prep, columns=feature_names)
print(new_df.T.round(3).to_string())

                        0
num__Age           -0.112
num__SibSp          0.492
num__Parch         -0.452
num__Fare          -0.122
cat__Pclass_First   0.000
cat__Pclass_Second  1.000
cat__Pclass_Third   0.000
cat__Sex_female     1.000
cat__Sex_male       0.000
cat__Embarked_C     0.000
cat__Embarked_Q     0.000
cat__Embarked_S     0.000


**Syntax in simple words:** The new passenger is a one-row table with the same columns as `X`. `transform` fills the missing age with the *training* median, scales with the *training* mean/std, and encodes categories using the *training* categories.

**Purpose:** This is exactly what happens when a model later meets real, unseen data.

**How to interpret the result:** `Embarked_*` are all 0 because 'X' is an unseen category and `handle_unknown='ignore'`; the age equals the scaled training median.

---
# GUIDED SELF-LEARNING 2 (Hour 7, 60 min)
## Section 13 — Compare strategies, debug, mini-challenges, conclusions
Hour 7 is independent work: finish any unfinished exercises first, then work through the tasks below.

### Exercise C6 — Compare two preprocessing strategies
**Task (Level 5, write it yourself):** Build a second numeric pipeline that fills missing values with the **mean** instead of the median. Fit both versions on `X_train` and compare, for `Age` and `Fare`: (a) the value used to fill missing numbers, (b) the transformed value of a passenger with a missing Fare. Which strategy would you defend, and why?

<details><summary>💡 Hint (try first without it)</summary>

Copy numeric_pipeline, change the strategy, wrap each in `skc.ColumnTransformer` (imported in C1) for numeric_features only, call `fit` on X_train. `imputer.statistics_` shows the learned fill values.

</details>

In [21]:
# ===== Exercise C6 - SOLUTION =====
def numeric_branch(strategy):
    pipe = Pipeline([("impute", SimpleImputer(strategy=strategy)), ("scale", StandardScaler())])
    return skc.ColumnTransformer([("num", pipe, numeric_features)]).fit(X_train)

median_version, mean_version = numeric_branch("median"), numeric_branch("mean")
fill = pd.DataFrame({"median": median_version.named_transformers_["num"].named_steps["impute"].statistics_,
                     "mean": mean_version.named_transformers_["num"].named_steps["impute"].statistics_}, index=numeric_features).round(2)
print(fill)
no_fare = pd.DataFrame([{"Age": 30, "SibSp": 0, "Parch": 0, "Fare": np.nan}])
print("Scaled Fare for a passenger with missing Fare -> median version: %.3f | mean version: %.3f" %
      (median_version.transform(no_fare)[0, 3], mean_version.transform(no_fare)[0, 3]))

       median   mean
Age     28.75  30.71
SibSp    0.00   0.50
Parch    0.00   0.38
Fare    14.45  33.07
Scaled Fare for a passenger with missing Fare -> median version: -0.339 | mean version: -0.000


**Trainer note (C6):** The mean Fare is much higher than the median because of a few very expensive tickets (and any unresolved extreme values), so a passenger with a missing Fare would be treated as paying far more than a typical passenger. For skewed columns the **median** is the safer default; for roughly symmetric columns the two nearly agree (see Age).

### Exercises D1 – D3 — Debugging (Level 4)
Each cell below has a bug. Predict what is wrong, then fix it so the check at the end passes.

**D1 — Leakage.** The learner fits the preprocessing on the **test** data. Fix it.

In [22]:
# ===== Exercise D1 (Level 4: DEBUG) - FIXED =====
# BUG: fit_transform(X_test) re-learns medians, means and categories from the TEST data (leakage), and train/test now use different rules.
# FIX: fit on the training data only, then only transform the test data.
from sklearn.base import clone
good_preprocess = clone(preprocess)
good_train = good_preprocess.fit_transform(X_train)
good_test = good_preprocess.transform(X_test)          # transform only

used = good_preprocess.named_transformers_["num"].named_steps["impute"].statistics_
train_medians = preprocess.named_transformers_["num"].named_steps["impute"].statistics_
assert np.allclose(used, train_medians), "The preprocessing was re-learned from the test data (leakage)!"
print("D1 fixed: test data never influenced the learned medians")

D1 fixed: test data never influenced the learned medians


**D2 — Crash on an unseen category.** This encoder works on training data but crashes on new data. Fix the encoder.

In [23]:
# ===== Exercise D2 (Level 4: DEBUG) - FIXED =====
# BUG: by default OneHotEncoder raises an error for a category it did not see in training.
# FIX: handle_unknown="ignore" encodes an unseen category as all zeros.
safe_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
safe_encoder.fit(X_train[["Embarked"]].fillna("S"))
new_ports = pd.DataFrame({"Embarked": ["S", "C", "X"]})
encoded_ports = safe_encoder.transform(new_ports)
print(encoded_ports)
assert encoded_ports[2].sum() == 0, "An unseen category should be all zeros"

[[0. 0. 1.]
 [1. 0. 0.]
 [0. 0. 0.]]


**D3 — Imputing before splitting.** This cell fills missing fares with the **mean of the whole modelling table**. Show that this uses test information, and replace it by the correct approach (a fitted imputer).

In [24]:
# ===== Exercise D3 (Level 4: DEBUG) - FIXED =====
# BUG: the mean of the whole table includes the test rows, so the test set influences preprocessing (leakage).
# FIX: let an imputer LEARN the value from the training rows only (fit on X_train), then reuse it.
whole_mean = model_df["Fare"].mean()
fitted = SimpleImputer(strategy="mean").fit(X_train[["Fare"]])
learned = fitted.statistics_[0]
print("Mean fare of all rows:", round(whole_mean, 4), "| value learned from training rows:", round(learned, 4))
assert learned == X_train["Fare"].mean(), "The fill value must be learned from the training rows only"
assert learned != whole_mean, "(sanity) the whole-table mean differs, so the test rows would have leaked"
print("D3 fixed: the imputer learns from training rows only")

Mean fare of all rows: 32.7386 | value learned from training rows: 33.0682
D3 fixed: the imputer learns from training rows only


**Trainer note (D1-D3):** D1 is the classic leakage bug; D2 is the classic production crash; D3 shows that a hand-computed mean before the split differs from the training mean (the difference is the leakage).

### Mini-challenges (MC1 – MC3, Level 5)
**MC1 — Effect of extreme fares.** Compare the **standard deviation learned by StandardScaler** for `Fare` in the training set with and without the 5 highest fares. How much does it change and what does that tell you about outliers and scaling?

**MC2 — Different strategy per column.** Build a pipeline where `Age` is filled with the **mean** and `Fare` with the **median** (two separate numeric branches). What are the two learned fill values?

**MC3 — Pipeline on quarantined rows.** Apply the fitted `preprocess` to the conflicting-ID rows you quarantined in A8 (after dropping the columns not used as features). Report the shape and whether any value is missing. Why is this an acceptable use of `transform`?

In [25]:
# ===== MC1 - SOLUTION =====
fare_train = X_train[["Fare"]].dropna()
top5_removed = fare_train.drop(fare_train["Fare"].nlargest(5).index)
std_all = StandardScaler().fit(fare_train).scale_[0]
std_wo = StandardScaler().fit(top5_removed).scale_[0]
print("Fare std learned: all training rows = %.2f | without the 5 highest = %.2f" % (std_all, std_wo))

# ===== MC2 - SOLUTION =====
two_branch = skc.ColumnTransformer([
    ("age", Pipeline([("impute", SimpleImputer(strategy="mean")), ("scale", StandardScaler())]), ["Age"]),
    ("fare", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), ["Fare"]),
]).fit(X_train)
print("Age fill (mean): %.2f | Fare fill (median): %.2f" % (two_branch.named_transformers_["age"].named_steps["impute"].statistics_[0],
                                                           two_branch.named_transformers_["fare"].named_steps["impute"].statistics_[0]))

# ===== MC3 - SOLUTION =====
quarantine_X = quarantine_conflicts.drop(columns=exclude + [target])
quarantine_prep = preprocess.transform(quarantine_X)
print("Quarantined rows transformed:", quarantine_prep.shape, "| any NaN left:", bool(np.isnan(quarantine_prep).any()))

Fare std learned: all training rows = 53.98 | without the 5 highest = 39.57
Age fill (mean): 30.71 | Fare fill (median): 14.45
Quarantined rows transformed: (12, 12) | any NaN left: False


**Trainer note (MC1-MC3):** MC1: the 5 highest genuine fares inflate the learned standard deviation noticeably, so most passengers get z-scores close together near 0: scaling does not remove outliers, it is *affected* by them. MC2: ColumnTransformer allows a different strategy per column. MC3: using `transform` on quarantined rows is legitimate because they are treated as unseen data; nothing is learned from them.

### Exercise C7 — Final decision log, raw-file check and reflection
**Part 1.** Complete the final Cleaning Decision Log (revise your plan from SL2 and A7 using what you learned).

| Problem | Final decision | Reason | Done before split or inside pipeline? | Remaining risk |
|---|---|---|---|---|
| Spelling, case, aliases | | | | |
| Text numbers | | | | |
| Impossible values | | | | |
| Missing numbers | | | | |
| Missing categories | | | | |
| Exact duplicates | | | | |
| Conflicting IDs | | | | |
| Identifier / mostly empty columns | | | | |
| Outliers kept (genuine fares, ages 90-100) | | | | |

**Part 2.** Write your final reflection (2–4 sentences each): *What did you diagnose? What did you decide and why? Where could leakage have happened and how did the pipeline prevent it? What remains unresolved? What will Day 3 build on?*

**Model final decision log (C7)**

| Problem | Final decision | Reason | Before split / pipeline | Remaining risk |
|---|---|---|---|---|
| Spelling, case, aliases | strip, lower, dictionary map | one value, one label | before | unmapped variants become missing |
| Text numbers | regex + `to_numeric` | enables arithmetic | before | coercion hides unreadable text |
| Impossible values | set to NaN (counted) | cannot be true | before | later filled by estimates |
| Missing numbers | median in pipeline | robust to skewed fares; learned from train only | **pipeline** | assumes missing at random |
| Missing categories | most frequent in pipeline | keeps rows; learned from train only | **pipeline** | favours the majority class |
| Exact duplicates | drop | no extra information | before | none |
| Conflicting IDs | quarantine all rows | one passenger must not appear in train and test | before | loses genuine rows |
| Identifier / empty columns | exclude | no generalisable information | before | Cabin might yield a feature in Day 3 |
| Outliers kept | keep and flag | genuine fares and ages | - | scaling sensitivity (see MC1) |

**Model reflection:** *"The classroom Titanic table had missing ages, fares and categories, spelling variants and abbreviations, text numbers (`22 yrs`, `£7.25`), impossible values, exact duplicates and conflicting repeated IDs. I made only rule-based fixes before the split and quarantined conflicting IDs and unlabelled rows so that nobody could appear in both training and test. Medians, most-frequent categories, means and category lists were learned inside a Pipeline from training data only, then applied to the test set and to a new passenger with `transform`; this prevented preprocessing leakage. Genuine outliers (first-class fares, infants, ages up to 80) were kept. Unresolved: the 2 suspicious ages, the quarantined rows and the treatment of Cabin and Name, which Day 3 (feature engineering) will address."*

In [26]:
# ===== Raw file check (provided) =====
import hashlib
assert hashlib.md5(open(DATA_FILE, "rb").read()).hexdigest() == RAW_FINGERPRINT, "Raw file was modified!"
print("Raw file untouched")

Raw file untouched


---
## ✅ Submission checklist
- [ ] All blanks completed; *Restart & Run All* works with outputs visible
- [ ] Prediction boxes, SL1–SL3, A7 table, C6, D1–D3, MC1–MC3 and C7 completed
- [ ] Final decision log and reflection written
- [ ] Raw CSV unchanged

**See you on Day 3: feature engineering, feature selection, PCA and t-SNE.** 🎓